# Z4 115M on Blackwell / CUDA

Same `kaggle10m_nomtp` profile as the measured T4 notebook:

- 115,238,237 parameters
- WikiText-2, ctx 512, batch 4
- 600-second training cap
- concept loop + feedback bus, MTP compute skipped
- Blackwell uses **bf16** autocast automatically (`--amp-dtype auto`); T4 stays fp16

This notebook is for a CUDA machine (Blackwell, H100, A100, or T4).  It is
not the Kaggle TPU path.  Run the cells in order.


In [ ]:
import json, os, subprocess, sys, urllib.request
from pathlib import Path

print(sys.version)
subprocess.run([
    sys.executable, "-c",
    "import torch; print('torch', torch.__version__); "
    "print('cuda', torch.version.cuda, torch.cuda.is_available()); "
    "print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None); "
    "print('capability', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)",
], check=False)
subprocess.run(["nvidia-smi"], check=False)


In [ ]:
REPO = None
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / "z3" / "train_v3.py").is_file():
        REPO = base
        break
if REPO is None:
    REPO = Path.cwd() / "nanlm_experimental"
    if not (REPO / "z3" / "train_v3.py").is_file():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/shemshallah/nanlm_experimental.git",
            str(REPO),
        ], check=True)
os.chdir(REPO)
print(REPO)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers", "pyarrow"], check=False)


In [ ]:
DATA = REPO / "data"
BINS = DATA / "wikitext_bins"
if not (BINS / "train.bin").is_file():
    DATA.mkdir(parents=True, exist_ok=True)
    parts = []
    for split in ("train", "valid"):
        dest = DATA / f"wikitext_{split}.txt"
        if not dest.is_file():
            url = ("https://raw.githubusercontent.com/pytorch/examples/main/"
                   f"word_language_model/data/wikitext-2/{split}.txt")
            urllib.request.urlretrieve(url, dest)
        parts.append(dest.read_text(encoding="utf-8"))
    (DATA / "wikitext_corpus.txt").write_text("\n\n".join(parts), encoding="utf-8")
    subprocess.run([
        sys.executable, "z3/build_domain_bins.py",
        "--corpus", "wikitext=data/wikitext_corpus.txt",
        "--out", "data/wikitext_bins",
        "--max-bytes-per-domain", "12000000",
        "--train-fraction", "0.98", "--valid-fraction", "0.01",
    ], check=True)
print(BINS)


In [ ]:
OUT = REPO / "runs" / "z4_115m_blackwell"
cmd = [
    sys.executable, "z3/train_v3.py",
    "--device", "cuda", "--amp", "--amp-dtype", "auto",
    "--preset", "kaggle10m_nomtp",
    "--corpus", "data/wikitext_bins/train.bin",
    "--val-corpus", "data/wikitext_bins/valid.bin",
    "--data-manifest", "data/wikitext_bins/manifest.json",
    "--batch-size", "4",
    "--ctx", "512",
    "--max-seconds", "600",
    "--no-grad-checkpoint",
    "--out", str(OUT),
]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
CKPT = OUT / "best.pt"
if not CKPT.is_file():
    CKPT = OUT / "final.pt"
print("evaluating", CKPT, flush=True)
subprocess.run([
    sys.executable, "z3/eval_domains.py",
    "--checkpoint", str(CKPT),
    "--test-dir", "data/wikitext_bins/test",
    "--data-manifest", "data/wikitext_bins/manifest.json",
    "--tokenizer", "layered", "--document-framing", "--sidecar", "--require-sidecar",
    "--ctx", "512", "--batch-size", "4", "--device", "cuda",
    "--out", str(OUT / "test_domains.json"),
], check=True)


In [ ]:
result = json.loads((OUT / "result.json").read_text())
test = json.loads((OUT / "test_domains.json").read_text())
best = (result.get("best") or {}).get("metrics") or {}
print(json.dumps({
    "params": result.get("params"),
    "steps": result.get("steps_completed"),
    "tokens_seen": result.get("tokens_seen"),
    "wall_seconds": result.get("wall_seconds"),
    "best_step": (result.get("best") or {}).get("step"),
    "best_val_bpb": best.get("bits_per_byte"),
    "final_val_bpb": (result.get("final") or {}).get("bits_per_byte"),
    "test_bpb": test.get("macro_bits_per_byte"),
    "nonfinite_skips": result.get("nonfinite_skips"),
}, indent=2, sort_keys=True))
